In [1]:
import fine as fn
import pyomo.environ as pyomo
import pandas as pd 

# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A", "B"},
    onlycommodities={"electricity", "hydrogen"},
    onlycommodityUnitsDict={"electricity": "GW", "hydrogen": "kg"},
    onlymaterials={"steel", "copper"},
    onlymaterialUnitsDict={"steel": "tons", "copper": "kg"}
)

In [2]:
# energyCommoditySet = {'electricty'}
# materialCommoditySet = {'steel'}
# commodityUnitDict = {'electricty': 'kWh', 'steel': 't'}
# print(processedMaterialIntensity[('location1', 2020, 'material1')]) 

# print(processedMaterialIntensity["location1"]["2020"]["material1"]) 

In [3]:
# Check if commodity declarations work
print("only Materials Units Dict:", esM.onlymaterialUnitsDict)
print("only Commodity Units Dict:", esM.onlycommodityUnitsDict)
print("Commodities:", esM.commodities)
print("Commodity Units Dict:", esM.commodityUnitsDict)

only Materials Units Dict: {'steel': 'tons', 'copper': 'kg'}
only Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg'}
Commodities: ['electricity', 'hydrogen', 'copper', 'steel']
Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg', 'copper': 'kg', 'steel': 'tons'}


In [4]:
# Step 2: Add a Energy Source Component that Requires Materials                             
esM.add(
    fn.Source(
        esM=esM, 
        name="Wind Turbines",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity = {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
        materialRecovery= {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
    )
)

# # Add a Energy Storage Component that Requires Materials
esM.add(
    fn.Source(
        esM=esM, 
        name="Wind (off shore)",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity = {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
        materialRecovery= {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
    )
)


In [5]:
# Step 3: Add Material Source 
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Recycling",
        commodity="steel",
        hasCapacityVariable=True,
        material=True,
    )
)

source = esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Recycling",
        commodity="copper",
        hasCapacityVariable=True,
        material=True,
    )
)
#esM.source.commodity

In [6]:
# Step 4: Add Energy Sink Component that consumes Energy 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=50,
        
    )
)

In [7]:
# Step 5: Add Material Sink that consumes Materials 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,      
    )
)

In [8]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(2.6107 sec)



In [9]:
# after you have your esM instance...

print("=== materialIntensity for all components ===")
for cm in esM.componentModelingDict.values():
    for compName, comp in cm.componentsDict.items():
        mi = getattr(comp, "materialIntensity", None)
        # skip anything that has no dict or is literally 0
        if not mi or mi == 0:
            continue

        print(f"\nComponent {compName!r} — materialIntensity:")
        # mi is typically a dict: { loc: { mat: pd.Series({ip: val, …}), … }, … }
        for loc, mat_dict in mi.items():
            for mat, series in mat_dict.items():
                # pandas.Series over investmentPeriods
                for ip, val in series.items():
                    print(f"  loc={loc}, material={mat}, ip={ip}  ->  {val}")


=== materialIntensity for all components ===

Component 'Wind Turbines' — materialIntensity:
  loc=A, material=steel, ip=0  ->  3.1
  loc=A, material=steel, ip=1  ->  3.2
  loc=A, material=copper, ip=0  ->  5.3
  loc=A, material=copper, ip=1  ->  3.2
  loc=B, material=steel, ip=0  ->  2.9
  loc=B, material=steel, ip=1  ->  3.0
  loc=B, material=copper, ip=0  ->  4.8
  loc=B, material=copper, ip=1  ->  4.9

Component 'Wind (off shore)' — materialIntensity:
  loc=A, material=steel, ip=0  ->  3.1
  loc=A, material=steel, ip=1  ->  3.2
  loc=A, material=copper, ip=0  ->  5.3
  loc=A, material=copper, ip=1  ->  3.2
  loc=B, material=steel, ip=0  ->  2.9
  loc=B, material=steel, ip=1  ->  3.0
  loc=B, material=copper, ip=0  ->  4.8
  loc=B, material=copper, ip=1  ->  4.9


In [10]:
esM.declareOptimizationProblem()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
LHS_sinks op_srcSnk[A,Steel demand,0,0,0] + op_srcSnk[A,Steel demand,0,0,1] + op_srcSnk[A,Steel demand,0,0,2] + op_srcSnk[A,Steel demand,0,0,3] + op_srcSnk[A,Steel demand,0,0,4] + op_srcSnk[A,Steel demand,0,0,5] + op_srcSnk[A,Steel demand,0,0,6] + op_srcSnk[A,Steel demand,0,0,7] + op_srcSnk[A,Steel demand,0,0,8] + op_srcSnk[A,Steel demand,0,0,9] + op_srcSnk[A,Steel demand,0,0,10] + op_srcSnk[A,Steel demand,0,0,11] + op_srcSnk[A,Steel demand,0,0,12] + op_srcSnk[A,Steel demand,0,0,13] + op_srcSnk[A,Steel demand,0,0,14] + op_srcSnk[A,Steel demand,0,0,15] + op_srcSnk[A,Steel demand,0,0,16] + op_srcSnk[A,Steel demand,0,0,17] + op_srcSnk[A,Steel demand,0,0,18] + op_srcSnk[A,Steel demand,0,0,19] + op_srcSnk[A,Steel demand,0,0,20] + op_srcSnk[A,Steel demand,0,0,21] + op_srcSnk[A,Steel demand,0,0,22] + op_srcSnk[A,Steel demand,0,0,23] + op_srcSnk[A,Steel demand,0

In [11]:
esM.optimize(solver="glpk")

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
LHS_sinks op_srcSnk[A,Steel demand,0,0,0] + op_srcSnk[A,Steel demand,0,0,1] + op_srcSnk[A,Steel demand,0,0,2] + op_srcSnk[A,Steel demand,0,0,3] + op_srcSnk[A,Steel demand,0,0,4] + op_srcSnk[A,Steel demand,0,0,5] + op_srcSnk[A,Steel demand,0,0,6] + op_srcSnk[A,Steel demand,0,0,7] + op_srcSnk[A,Steel demand,0,0,8] + op_srcSnk[A,Steel demand,0,0,9] + op_srcSnk[A,Steel demand,0,0,10] + op_srcSnk[A,Steel demand,0,0,11] + op_srcSnk[A,Steel demand,0,0,12] + op_srcSnk[A,Steel demand,0,0,13] + op_srcSnk[A,Steel demand,0,0,14] + op_srcSnk[A,Steel demand,0,0,15] + op_srcSnk[A,Steel demand,0,0,16] + op_srcSnk[A,Steel demand,0,0,17] + op_srcSnk[A,Steel demand,0,0,18] + op_srcSnk[A,Steel demand,0,0,19] + op_srcSnk[A,Steel demand,0,0,20] + op_srcSnk[A,Steel demand,0,0,21] + op_srcSnk[A,Steel demand,0,0,22] + op_srcSnk[A,Steel demand,0,0,23] + op_srcSnk[A,Steel demand,0

In [12]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

A         B
Component          Property      Unit                          
Electricity demand operation     [GW*h/a]    438000.0  438000.0
                                 [GW*h]      438000.0  438000.0
Steel Recycling    capacity      [tons]          77.5      72.5
                   commissioning [tons]          77.5      72.5
                   operation     [tons*h/a]     155.0     145.0
                                 [tons*h]       155.0     145.0
Steel demand       operation     [tons*h/a]     155.0     145.0
                                 [tons*h]       155.0     145.0
Wind (off shore)   capacity      [GW]            50.0      50.0
                   commissioning [GW]            50.0      50.0
                   operation     [GW*h/a]    438000.0  438000.0
                                 [GW*h]      438000.0  438000.0

------------------------------------------------------------------------------------------------------------------------
## Tests